# DentalSelfie 7.2: Ordinal and Combined Method

## Research question

This notebook tests ordinal learning on the local-only control and on the gated paired-view model. CORAL targets encode ordered grades (no-visible < mild < marked); the combined model tests whether context fusion and ordinal supervision complement one another. These are literature-derived methods adapted to this dataset, not a claim of a new clinical algorithm.

The task is **three-class gingival appearance grading on annotation-derived ROIs**. It is a research prototype and does not diagnose disease. The locked validation split selects checkpoints; the held-out test split remains unopened. Dataset version: `trihuynhviprovcl/dentalselfie-gum-roi-v1/1`.

**Primary outcome:** validation macro-F1. Report balanced accuracy, per-class recall, confusion matrix and (for ordinal runs) MAE and quadratic weighted kappa. Each experiment is independent and can be resumed from its epoch checkpoint.

## Experiment matrix

| ID | Inputs | Fusion / objective | GPU |
|---|---|---|---|
| M3 | Local only | CORAL ordinal loss | T4 0 |
| M4 | Local + context | Learned feature gate + CORAL | T4 1 |

```mermaid
flowchart LR
  A[Locked paired ROI manifest] --> B[Verify classes, paths and split]
  B --> C[Shared ImageNet ResNet18 features]
  C --> D[Local CORAL vs gated CORAL]
  D --> E[Two independent T4 workers]
  E --> F[Validation metrics, predictions, checkpoints]
```

CORAL follows the cumulative ordinal formulation of Cao et al. (2020), implemented here with two monotone thresholds. Gated local/global feature fusion is inspired by recent image-modeling work such as DVCTNet (Luo et al., 2025); this is an adaptation to paired intraoral photographs, not a reproduction. The comparison is exploratory and may show no gain.

References: [CORAL paper](https://doi.org/10.1016/j.patrec.2020.11.008) - [CORAL PyTorch reference](https://github.com/Raschka-research-group/coral-pytorch) - [DVCTNet preprint](https://arxiv.org/abs/2508.20813).

**Runtime guard:** up to 20 epochs, early stop after 5 stale epochs, and 70 minutes per experiment. Every epoch saves latest state, best weights, history and prediction artifacts. If Kaggle stops a session, rerun the notebook and it resumes the matching experiment. Training is enabled for this authorized Kaggle submission.

## 1. Configure the run

Pin the manifest fingerprint, image normalization, effective batch size and per-worker bounds. GPU 0 and GPU 1 receive one experiment each; this keeps the comparison parallel without multiplying work on a single card.

In [ ]:
from pathlib import Path
import os, json, hashlib, random, time, copy, threading, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import functional as TF, InterpolationMode

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32                 # Per GPU; accumulate 2 steps = effective batch 64
GRAD_ACCUM_STEPS = 2
MAX_EPOCHS = 20                 # Bounded first screening; resume from saved checkpoints
PATIENCE = 5
MAX_SECONDS_PER_EXPERIMENT = 4200
NUM_WORKERS = 2
RUN_TRAINING = True
ROI_ROOTS = list(Path('/kaggle/input').glob('**/roi/manifest.csv'))
if not ROI_ROOTS:
    ROI_ROOTS = list(Path('/kaggle/input').glob('**/manifest.csv'))
if not ROI_ROOTS:
    raise FileNotFoundError('Attach trihuynhviprovcl/dentalselfie-gum-roi-v1 version 1.')
ROI_ROOT = ROI_ROOTS[0].parent
OUTPUT_DIR = Path('/kaggle/working/dentalselfie_07')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_SHA256 = '8ed4c05240a2d23e6c254ee74adf13af48ff45a4418f1e2024bd38498b02cab1'
CLASS_NAMES = ['no_visible_inflammation', 'mild_inflammation', 'marked_inflammation']
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
DEVICE_COUNT = torch.cuda.device_count()
print(f'ROI root: {ROI_ROOT}')
print(f'GPU count: {DEVICE_COUNT}; devices: {[torch.cuda.get_device_name(i) for i in range(DEVICE_COUNT)]}')
print(f'Max experiment wall budget: {MAX_SECONDS_PER_EXPERIMENT / 60:.0f} minutes')

## 2. Verify inputs and preserve the locked split

The manifest hash binds this run to the audited ROI release. Checks confirm class mapping, unique crop IDs, train/validation-only rows and all paired file paths. No test image is read.

In [ ]:
manifest_path = ROI_ROOT / 'manifest.csv'
manifest_bytes = manifest_path.read_bytes()
manifest_sha = hashlib.sha256(manifest_bytes).hexdigest()
assert manifest_sha == MANIFEST_SHA256, f'Manifest mismatch: {manifest_sha}'
manifest = pd.read_csv(manifest_path)
required = {'crop_id','parent_id','split','class_id','class_name','local_path','context_path','local_sha256','context_sha256'}
assert required.issubset(manifest.columns), required - set(manifest.columns)
assert set(manifest['split']) == {'train','val'}, set(manifest['split'])
assert np.array_equal(manifest['class_name'].map({n:i for i,n in enumerate(CLASS_NAMES)}).to_numpy(), manifest['class_id'].to_numpy())
assert manifest['crop_id'].is_unique
for column in ['local_path','context_path']:
    missing = [p for p in manifest[column] if not (ROI_ROOT / p).is_file()]
    assert not missing, f'{len(missing)} missing files in {column}'
print(f'Rows: {len(manifest):,}; parents: {manifest.parent_id.nunique():,}; manifest SHA-256: {manifest_sha}')
display(manifest.groupby(['split','class_id','class_name']).size().rename('ROI count').unstack(fill_value=0))

## 3. Inspect the data distribution

Counts reveal the severe class imbalance that makes accuracy alone misleading. The validation counts are descriptive and are not used to rebalance training.

In [ ]:
counts = manifest.groupby(['split','class_name']).size().unstack(fill_value=0).reindex(columns=CLASS_NAMES)
ax=counts.plot(kind='bar',figsize=(9,4),color=['#7aa6c2','#e4b45c','#bd6b67'])
ax.set_ylabel('ROI count'); ax.set_title('Locked ROI class counts by split'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 4. Inspect paired local and context views

The local crop focuses on the annotated gingival region. The context view keeps nearby oral anatomy. Showing both helps explain why context may help or add distracting information.

In [ ]:
fig, axes = plt.subplots(3,2,figsize=(8,10))
for row_i, class_id in enumerate(range(3)):
    row=manifest[(manifest.split=='train') & (manifest.class_id==class_id)].iloc[0]
    for col,path_col,title in [(0,'local_path','Local ROI'),(1,'context_path','Context crop')]:
        axes[row_i,col].imshow(Image.open(ROI_ROOT/row[path_col]).convert('RGB')); axes[row_i,col].set_title(f'{CLASS_NAMES[class_id]} - {title}'); axes[row_i,col].axis('off')
plt.tight_layout(); plt.show()

## 5. Build synchronized paired preprocessing

Both views receive the same random horizontal flip and rotation so geometric augmentation remains aligned. Resize to 224 x 224, convert RGB pixels to tensors, and normalize with ImageNet statistics. Augmentation is training-only.

In [ ]:
class PairedROIDataset(Dataset):
    def __init__(self, frame, root, training=False):
        self.frame = frame.reset_index(drop=True)
        self.root = Path(root)
        self.training = training
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(MEAN, STD)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        local = Image.open(self.root / row.local_path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
        context = Image.open(self.root / row.context_path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
        if self.training:
            if random.random() < 0.5:
                local, context = TF.hflip(local), TF.hflip(context)
            angle = random.uniform(-7, 7)
            local = TF.rotate(local, angle, interpolation=InterpolationMode.BILINEAR, fill=(128,128,128))
            context = TF.rotate(context, angle, interpolation=InterpolationMode.BILINEAR, fill=(128,128,128))
        local = self.normalize(self.to_tensor(local))
        context = self.normalize(self.to_tensor(context))
        return local, context, int(row.class_id), str(row.crop_id)

train_frame = manifest[manifest.split == 'train'].copy()
val_frame = manifest[manifest.split == 'val'].copy()
train_ds = PairedROIDataset(train_frame, ROI_ROOT, training=True)
val_ds = PairedROIDataset(val_frame, ROI_ROOT, training=False)
print(f'Train pairs: {len(train_ds):,}; validation pairs: {len(val_ds):,}')

## 6. Form batches and inspect the model inputs

Batch size 32 is used on each T4. Gradient accumulation over two batches gives an effective batch size of 64 while training two independent experiments in parallel.

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
local_batch, context_batch, label_batch, id_batch = next(iter(train_loader))
print('Local tensor:', tuple(local_batch.shape), '| context tensor:', tuple(context_batch.shape), '| labels:', tuple(label_batch.shape))
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, tensor, title in zip(axes, [local_batch[0], context_batch[0]], ['Local ROI', 'Context crop']):
    image = tensor.permute(1,2,0).numpy() * np.array(STD) + np.array(MEAN)
    ax.imshow(np.clip(image,0,1)); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Define the shared ResNet18 and method-specific heads

A single ImageNet-pretrained ResNet18 encoder extracts 512-dimensional features. M1 concatenates local/context vectors; M2 uses a feature-wise sigmoid gate to blend them. In 7.2, the same encoder is paired with the local-only or gated head and a CORAL ordinal output.

### Method-specific output and shape check

The smoke forward pass verifies the tensor contract before workers start. CORAL returns two cumulative logits; cross-entropy returns three class logits.

In [ ]:
METHOD = "gate"
ORDINAL = True
from torchvision.models import resnet18, ResNet18_Weights

class PairedResNet(nn.Module):
    """Shared ImageNet ResNet18 encoder with one declared fusion/head variant."""
    def __init__(self, method, ordinal=False):
        super().__init__()
        self.method = method
        self.ordinal = ordinal
        base = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.feature_dim = 512
        if method == 'concat':
            self.fusion = nn.Sequential(nn.Linear(1024,512), nn.ReLU(inplace=True), nn.Dropout(0.25))
            self.gate = None
        elif method == 'gate':
            self.gate = nn.Sequential(nn.Linear(1024,256), nn.ReLU(inplace=True), nn.Linear(256,512), nn.Sigmoid())
            self.fusion = nn.Dropout(0.25)
        elif method == 'local':
            self.gate = None
            self.fusion = nn.Identity()
        else:
            raise ValueError(method)
        output_dim = 2 if ordinal else 3
        self.head = nn.Linear(512, output_dim)
        self.threshold_gap = nn.Parameter(torch.tensor(0.0)) if ordinal else None
    def forward(self, local, context=None, return_gate=False):
        f_local = self.encoder(local).flatten(1)
        if self.method == 'local':
            fused = self.fusion(f_local)
            gate = None
        else:
            if context is None: raise ValueError('This method requires paired context images.')
            f_context = self.encoder(context).flatten(1)
            if self.method == 'concat':
                fused = self.fusion(torch.cat([f_local, f_context], dim=1)); gate = None
            else:
                gate = self.gate(torch.cat([f_local, f_context], dim=1))
                fused = self.fusion(gate * f_local + (1.0 - gate) * f_context)
        scores = self.head(fused)
        if self.ordinal:
            # Ordered thresholds guarantee P(y > 0) >= P(y > 1).
            gap = F.softplus(self.threshold_gap) + 1e-4
            thresholds = torch.stack([scores[:,0], scores[:,0] - gap], dim=1)
            scores = thresholds
        if return_gate:
            return scores, gate
        return scores

def coral_targets(labels):
    return torch.stack([(labels > 0).float(), (labels > 1).float()], dim=1)

def coral_loss(logits, labels):
    targets = coral_targets(labels)
    return F.binary_cross_entropy_with_logits(logits, targets, reduction='none').sum(dim=1).mean()

def decode_coral(logits):
    return (torch.sigmoid(logits) > 0.5).sum(dim=1)

smoke = PairedResNet(METHOD, ordinal=ORDINAL)
with torch.no_grad():
    out = smoke(local_batch[:2], context_batch[:2] if METHOD != 'local' else None)
print('Model:', METHOD, '| ordinal:', ORDINAL, '| output shape:', tuple(out.shape), '| parameters:', f'{sum(p.numel() for p in smoke.parameters()):,}')
del smoke

## 8. Train both experiments concurrently

Each worker owns one GPU, model, optimizer and checkpoint folder. Validation macro-F1 drives the scheduler and early stopping. Mixed precision and gradient accumulation reduce T4 memory pressure. The cell writes per-experiment curves, resumable states, best weights, validation predictions and machine-readable metrics.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, confusion_matrix, cohen_kappa_score
from concurrent.futures import ThreadPoolExecutor, as_completed
import csv

EXPERIMENTS = [{"id":"M3_local_CORAL","method":"local","ordinal":True},{"id":"M4_gate_CORAL","method":"gate","ordinal":True}]

def fit_one(experiment, gpu_id):
    torch.cuda.set_device(gpu_id)
    device = torch.device(f'cuda:{gpu_id}')
    seed = SEED + gpu_id
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    model = PairedResNet(experiment['method'], ordinal=experiment['ordinal']).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    scaler = torch.amp.GradScaler('cuda')
    best_f1, stale, start_epoch = -1.0, 0, 1
    history = []
    run_dir = OUTPUT_DIR / experiment['id']; run_dir.mkdir(parents=True, exist_ok=True)
    best_path, last_path = run_dir/'best.pt', run_dir/'last.pt'
    resume_path = run_dir/'resume.pt'
    if resume_path.exists():
        state = torch.load(resume_path, map_location=device, weights_only=False)
        model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer']); scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler']); start_epoch = state['epoch'] + 1
        best_f1 = state['best_f1']; stale = state['stale']; history = state['history']
        print(f"[{experiment['id']}] Resuming at epoch {start_epoch}")
    started = time.time()
    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        if time.time() - started > MAX_SECONDS_PER_EXPERIMENT:
            print(f"[{experiment['id']}] Time budget reached; checkpoint is resumable."); break
        model.train(); running_loss=0.0; optimizer.zero_grad(set_to_none=True)
        for step, (local, context, labels, _) in enumerate(train_loader, start=1):
            local=local.to(device, non_blocking=True); context=context.to(device, non_blocking=True); labels=labels.to(device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits=model(local, context if experiment['method'] != 'local' else None)
                loss = coral_loss(logits, labels) if experiment['ordinal'] else F.cross_entropy(logits, labels)
                scaled_loss=loss / GRAD_ACCUM_STEPS
            scaler.scale(scaled_loss).backward()
            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
            running_loss += float(loss.detach())
        model.eval(); truth=[]; pred=[]; val_loss=0.0
        with torch.no_grad():
            for local, context, labels, _ in val_loader:
                local=local.to(device, non_blocking=True); context=context.to(device, non_blocking=True); labels=labels.to(device, non_blocking=True)
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    logits=model(local, context if experiment['method'] != 'local' else None)
                    loss=coral_loss(logits, labels) if experiment['ordinal'] else F.cross_entropy(logits, labels)
                prediction=decode_coral(logits) if experiment['ordinal'] else logits.argmax(1)
                val_loss += float(loss)*len(labels); truth.extend(labels.cpu().tolist()); pred.extend(prediction.cpu().tolist())
        macro=f1_score(truth,pred,average='macro',zero_division=0); accuracy=accuracy_score(truth,pred); bal=balanced_accuracy_score(truth,pred)
        scheduler.step(macro)
        row={'epoch':epoch,'train_loss':running_loss/max(1,len(train_loader)),'val_loss':val_loss/len(val_ds),'accuracy':accuracy,'macro_f1':macro,'balanced_accuracy':bal,'lr':optimizer.param_groups[0]['lr'],'seconds':time.time()-started}
        history.append(row); print(f"[{experiment['id']}] epoch {epoch:02d}/{MAX_EPOCHS} loss={row['train_loss']:.4f}/{row['val_loss']:.4f} macroF1={macro:.4f} balAcc={bal:.4f} sec={row['seconds']:.0f}")
        state={'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'epoch':epoch,'best_f1':best_f1,'stale':stale,'history':history,'experiment':experiment,'seed':seed,'manifest_sha256':MANIFEST_SHA256}
        if macro > best_f1:
            best_f1=macro; stale=0; torch.save(model.state_dict(),best_path)
        else: stale += 1
        state.update({'best_f1':best_f1,'stale':stale}); torch.save(state,resume_path); torch.save(model.state_dict(),last_path)
        pd.DataFrame(history).to_csv(run_dir/'history.csv',index=False)
        if stale >= PATIENCE: print(f"[{experiment['id']}] Early stopping after {epoch} epochs."); break
    if best_path.exists():
        model.load_state_dict(torch.load(best_path,map_location=device,weights_only=True)); model.eval()
    truth=[]; pred=[]; probability=[]; gate_values=[]; ids=[]
    with torch.no_grad():
        for local,context,labels,crop_ids in val_loader:
            local=local.to(device); context=context.to(device); labels=labels.to(device)
            with torch.autocast(device_type='cuda',dtype=torch.float16):
                result=model(local,context if experiment['method']!='local' else None,return_gate=True)
            logits,gate=result
            prediction=decode_coral(logits) if experiment['ordinal'] else logits.argmax(1)
            probs=torch.sigmoid(logits) if experiment['ordinal'] else torch.softmax(logits,dim=1)
            truth.extend(labels.cpu().tolist()); pred.extend(prediction.cpu().tolist()); probability.extend(probs.float().cpu().tolist()); ids.extend(crop_ids)
            if gate is not None: gate_values.extend(gate.float().mean(dim=1).cpu().tolist())
    metrics={'experiment':experiment['id'],'method':experiment['method'],'ordinal':experiment['ordinal'],'gpu':torch.cuda.get_device_name(gpu_id),'epochs':len(history),'selected_epoch':int(max(history,key=lambda x:x['macro_f1'])['epoch']) if history else 0,'runtime_seconds':round(time.time()-started,1),'accuracy':accuracy_score(truth,pred),'macro_f1':f1_score(truth,pred,average='macro',zero_division=0),'balanced_accuracy':balanced_accuracy_score(truth,pred),'ordinal_mae':float(np.mean(np.abs(np.array(truth)-np.array(pred)))),'quadratic_kappa':cohen_kappa_score(truth,pred,weights='quadratic'),'confusion_matrix':confusion_matrix(truth,pred,labels=[0,1,2]).tolist(),'class_recall':(confusion_matrix(truth,pred,labels=[0,1,2]).diagonal()/np.maximum(1,confusion_matrix(truth,pred,labels=[0,1,2]).sum(axis=1))).tolist(),'gate_mean':float(np.mean(gate_values)) if gate_values else None,'manifest_sha256':MANIFEST_SHA256}
    with open(run_dir/'validation_predictions.csv','w',newline='',encoding='utf-8') as f:
        writer=csv.writer(f); writer.writerow(['crop_id','true_class','predicted_class','probabilities']); writer.writerows(zip(ids,truth,pred,probability))
    with open(run_dir/'metrics.json','w',encoding='utf-8') as f: json.dump(metrics,f,indent=2)
    return metrics

if RUN_TRAINING:
    if DEVICE_COUNT < 2: raise RuntimeError(f'T4 x2 required by this run; found {DEVICE_COUNT} GPU(s).')
    results=[]
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures=[pool.submit(fit_one, exp, gpu) for exp,gpu in zip(EXPERIMENTS,[0,1])]
        for future in as_completed(futures): results.append(future.result())
    results=sorted(results,key=lambda x:x['experiment'])
    pd.DataFrame([{k:v for k,v in row.items() if k!='confusion_matrix'} for row in results]).to_csv(OUTPUT_DIR/'summary.csv',index=False)
    with open(OUTPUT_DIR/'summary.json','w',encoding='utf-8') as f: json.dump(results,f,indent=2)
    display(pd.DataFrame([{k:v for k,v in row.items() if k not in ['confusion_matrix','class_recall']} for row in results]))

## 9. Compare with the completed C2 baseline

C2 is the measured local-only ImageNet ResNet18 result from notebook 6.3. It is a reference point, not a matched repeat: the new runs use paired-view preprocessing and the bounded screening schedule. Do not infer significance from a single seed.

In [ ]:
C2_BASELINE = {"accuracy": 0.6958333, "macro_f1": 0.5575762, "balanced_accuracy": 0.5470708}
summary_path=OUTPUT_DIR/'summary.csv'
if summary_path.exists():
    summary=pd.read_csv(summary_path)
    display(summary[['experiment','method','accuracy','macro_f1','balanced_accuracy','runtime_seconds']])
    fig,ax=plt.subplots(figsize=(8,4));
    labels=['C2 local baseline']+summary.experiment.tolist(); values=[C2_BASELINE['macro_f1']]+summary.macro_f1.tolist()
    ax.bar(labels,values,color=['#888888']+['#3572A5']*len(summary)); ax.set_ylabel('Validation macro-F1'); ax.set_ylim(0,1); ax.set_title('Exploratory validation comparison'); plt.xticks(rotation=15); plt.tight_layout(); plt.show()
else: print('Run the training cell to create experiment results.')

## 10. Inspect confusion matrices and class recalls

Read these together: a strong majority-class score can hide missed mild or no-visible cases. All results are validation-only and are not clinical performance claims.

In [ ]:
if (OUTPUT_DIR/'summary.json').exists():
    results=json.loads((OUTPUT_DIR/'summary.json').read_text(encoding='utf-8'))
    for result in results:
        print(result['experiment'], '| macro-F1',round(result['macro_f1'],4),'| recall',np.round(result['class_recall'],3))
        display(pd.DataFrame(result['confusion_matrix'],index=CLASS_NAMES,columns=CLASS_NAMES))
else: print('Metrics appear after training completes.')

## 11. Review learning curves and saved artifacts

Curves show convergence and overfitting. Checkpoint fingerprints include the manifest hash and experiment settings; keep the Kaggle output bundle with the public notebook version.

In [ ]:
for exp in EXPERIMENTS:
    p=OUTPUT_DIR/exp['id']/'history.csv'
    if p.exists():
        h=pd.read_csv(p); fig,axes=plt.subplots(1,2,figsize=(10,3.5)); axes[0].plot(h.epoch,h.train_loss,label='train'); axes[0].plot(h.epoch,h.val_loss,label='validation'); axes[0].legend(); axes[0].set_title(exp['id']+' loss'); axes[1].plot(h.epoch,h.macro_f1,marker='o'); axes[1].set_title('Validation macro-F1'); axes[1].set_xlabel('Epoch'); plt.tight_layout(); plt.show()
print('Output files:'); [print(p.relative_to(OUTPUT_DIR)) for p in sorted(OUTPUT_DIR.rglob('*')) if p.is_file()]

## 12. Interpretation and next decision

For 7.1, compare concat and gate against C2 to assess the value of paired context and adaptive weighting. For 7.2, compare local CORAL with gated CORAL to separate ordinal supervision from context fusion. A gain is only a hypothesis until repeated with additional seeds and a frozen test evaluation. Preserve all failed and resumed runs in the record.